# Phase 3 — Inference Parameter Optimization

Reproduces **Methods 2.4.3** and **Results 3.3** of *Automated Segmentation and Morphometric Assessment of Neuronal Somata in the Rat Trigeminal Ganglion Using a Fine-Tuned Cellpose-SAM Model*.

1. `diameter` and `min_size` are derived from the ground-truth morphology (`skimage.measure.regionprops`) of the independent validation set: mean equivalent diameter and smallest object area.
2. For each of the 10 Phase 2 models, a grid over `flow_threshold` (0.1–0.6) × `cellprob_threshold` (−2 to +2) is evaluated on the validation set; the combination with the highest AP@50 is selected (ties: AP@75, then F1).
3. Each model is re-evaluated on its own cross-validation test fold with the selected parameters.
4. Conditions A and B are compared with the one-standard-error rule, and the change relative to Phase 2 (default inference) is computed.

**Notebook series** (run in this order; each notebook reads the outputs of the previous ones):

| # | Notebook | Manuscript |
|---|----------|------------|
| 1 | `Phase1_training_hyperparameter_exploration.ipynb` | Methods 2.4.1 · Results 3.1 |
| 2 | `Phase2_generalization_data_augmentation.ipynb` | Methods 2.4.2 · Results 3.2 |
| 3 | `Phase3_inference_parameter_optimization.ipynb` | Methods 2.4.3 · Results 3.3 |
| 4 | `Phase4_morphology_measurements.ipynb` | Methods 2.5 · Results 3.4 · Table 1 · Figures 8–9 |
| 5 | `Phase5_figures.ipynb` | Figures 2–7 (no GPU needed) |

## Required files

| Path variable | Content | Provided file |
|---------------|---------|---------------|
| `VALIDATION_DIR` | 5 validation images (`3.2.2`, `Valid1`–`Valid4`) and annotations (`<id>_seg.npy`) | `dataset/validation/` |
| `CV_DIRS` | The 26 cross-validation images and annotations | `dataset/train/`, `dataset/test/` |
| `MODELS_DIR` | The 10 Phase 2 models (`<condition>/<condition>_fold_<k>_*.pth`) | `models/` |
| `PHASE2_DIR` | `A_no_augmentation/results_final.json`, `B_with_augmentation/results_final.json` (fold test images and Phase 2 metrics) | `results/phase2/` |

## Outputs (`RESULTS_DIR = PROJECT_DIR/results/phase3`)

| File | Content |
|------|---------|
| `phase3_validation_morphology.json` | `diameter`, `min_size` and validation morphology summary |
| `phase3_validation_gridsearch.json` | Full grid per model and selected parameters (input of `Phase5_figures`, Figure 6) |
| `phase3_validation_grid.csv` | Grid in long format |
| `phase3_test_evaluation.json` | Test-fold metrics with the selected parameters (input of Phase 4 and `Phase5_figures`, Figure 7) |
| `phase3_augmentation_summary.csv` | Mean, SD and SE per condition |
| `phase3_vs_phase2_delta.csv` | Per-fold change relative to Phase 2 |

**Expected folder layout** (files are distributed separately from the notebooks; set `PROJECT_DIR` in the *Paths* cell to the folder that contains them):

```
PROJECT_DIR/
├── dataset/
│   ├── train/        21 images <id>.tif + Cellpose annotations <id>_seg.npy
│   ├── test/          5 images (1, 7, 10, 15, 17) + <id>_seg.npy
│   └── validation/    5 images (3.2.2, Valid1–Valid4) + <id>_seg.npy
├── models/
│   ├── A_no_augmentation/     A_no_augmentation_fold_{1..5}_ep100_lr1e-05_wd0.1.pth
│   └── B_with_augmentation/   B_with_augmentation_fold_{1..5}_ep100_lr1e-05_wd0.1.pth
├── results/
│   ├── phase1/  phase2/  phase3/  phase4/
└── figures/
```

**Environment used in the study:** Google Colab Pro, NVIDIA RTX PRO 6000 Blackwell Server Edition GPU, Python 3, `cellpose==4.2.1.1` (Cellpose-SAM), PyTorch 2.11 (CUDA 12.8). Fixed seed: `RANDOM_SEED = 42`.

## 1. Environment

In [ ]:
!pip install --quiet "cellpose==4.2.1.1" scikit-image tifffile

In [ ]:
import gc
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile
import torch
from skimage.measure import regionprops
from tqdm.auto import tqdm

import cellpose
from cellpose import models
from cellpose import metrics as cp_metrics

print(f"cellpose {cellpose.version} | torch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

## 2. Paths

Only this cell needs to be edited. On Google Colab, mount Google Drive first (`from google.colab import drive; drive.mount('/content/drive')`) and point `PROJECT_DIR` to the folder that contains `dataset/`, `models/` and `results/`.

In [ ]:
PROJECT_DIR = Path(".")

VALIDATION_DIR = PROJECT_DIR / "dataset" / "validation"
CV_DIRS = [PROJECT_DIR / "dataset" / "train", PROJECT_DIR / "dataset" / "test"]
MODELS_DIR = PROJECT_DIR / "models"
PHASE2_DIR = PROJECT_DIR / "results" / "phase2"
RESULTS_DIR = PROJECT_DIR / "results" / "phase3"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 3. Parameters (fixed by the study design)

In [ ]:
FLOW_THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
CELLPROB_THRESHOLDS = [-2.0, -1.0, 0.0, 1.0, 2.0]

CONDITIONS = ["A_no_augmentation", "B_with_augmentation"]
TRAIN_HP = dict(n_epochs=100, learning_rate=1e-5, weight_decay=0.1, batch_size=1)
IOU_THRESHOLDS = [0.5, 0.75, 0.9]
CHANNEL_AXIS = -1
EPS = 1e-9

## 4. Helper functions

Metric convention identical to Phases 1–2: AP, TP, FP, FN from `cellpose.metrics.average_precision`; per-image Precision/Recall/F1 from each image's counts; aggregate AP = mean of per-image AP; aggregate Precision/Recall/F1 from pooled TP/FP/FN.

In [ ]:
IMAGE_EXTS = (".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp")


def load_image_rgb(path):
    img = np.asarray(tifffile.imread(str(path)))
    if img.ndim == 2:
        img = np.stack([img, img, img], axis=-1)
    elif img.ndim == 3:
        if img.shape[0] in (3, 4) and img.shape[-1] not in (3, 4):
            img = np.moveaxis(img, 0, -1)
        if img.shape[-1] == 4:
            img = img[..., :3]
    return np.ascontiguousarray(img)


def load_masks(path):
    arr = np.load(str(path), allow_pickle=True)
    try:
        d = arr.item()
        if isinstance(d, dict) and "masks" in d:
            return np.asarray(d["masks"]).astype(np.int32)
    except Exception:
        pass
    return np.asarray(arr).astype(np.int32)


def build_validation_index(folder):
    folder = Path(folder)
    imgs = sorted(p for p in folder.iterdir()
                  if p.is_file() and p.suffix.lower() in IMAGE_EXTS
                  and "_seg" not in p.stem and "_masks" not in p.stem)
    img_by_id, gt_by_id = {}, {}
    for p in imgs:
        cand = list(folder.glob(f"{p.stem}_seg.npy")) or list(folder.glob(f"{p.stem}*seg*.npy"))
        if cand:
            img_by_id[p.stem], gt_by_id[p.stem] = p, cand[0]
        else:
            print(f"WARNING: no _seg.npy for {p.name}")
    ids = sorted(img_by_id)
    print(f"Validation images paired with annotations: {ids}")
    return ids, img_by_id, gt_by_id


def resolve_cv_pair(image_id, folders):
    for folder in folders:
        img = next((p for p in Path(folder).iterdir()
                    if p.stem == str(image_id) and p.suffix.lower() in IMAGE_EXTS), None)
        if img is None:
            continue
        gt = None
        for pat in (f"{image_id}_seg.npy", f"{image_id}_masks.npy", f"{image_id}*seg*.npy"):
            c = list(Path(folder).glob(pat))
            if c:
                gt = c[0]
                break
        return img, gt
    return None, None


def compute_metrics(gt_list, pred_list, thresholds=IOU_THRESHOLDS, eps=EPS):
    ap, tp, fp, fn = cp_metrics.average_precision(gt_list, pred_list, threshold=thresholds)
    ap, tp, fp, fn = map(np.atleast_2d, (ap, tp, fp, fn))
    per_image, aggregate = {}, {}
    for ti, thr in enumerate(thresholds):
        key = f"AP@{thr}"
        P = tp[:, ti] / (tp[:, ti] + fp[:, ti] + eps)
        R = tp[:, ti] / (tp[:, ti] + fn[:, ti] + eps)
        F1 = 2 * P * R / (P + R + eps)
        per_image[key] = dict(AP=ap[:, ti].tolist(), TP=tp[:, ti].astype(int).tolist(),
                              FP=fp[:, ti].astype(int).tolist(), FN=fn[:, ti].astype(int).tolist(),
                              Precision=P.tolist(), Recall=R.tolist(), F1=F1.tolist())
        TP, FP, FN = int(tp[:, ti].sum()), int(fp[:, ti].sum()), int(fn[:, ti].sum())
        Pa = TP / (TP + FP + eps)
        Ra = TP / (TP + FN + eps)
        aggregate[key] = dict(AP=float(ap[:, ti].mean()), TP=TP, FP=FP, FN=FN,
                              Precision=Pa, Recall=Ra, F1=2 * Pa * Ra / (Pa + Ra + eps))
    return aggregate, per_image


def to_py(o):
    if isinstance(o, dict):
        return {k: to_py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_py(v) for v in o]
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.ndarray):
        return to_py(o.tolist())
    return o


def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(to_py(obj), f, indent=4)
    print("saved:", Path(path).name)

## 5. `diameter` and `min_size` from the validation ground truth

Expected values (manuscript): mean equivalent diameter = 104.5 px, smallest annotated soma = 1354 px².

In [ ]:
val_ids, VAL_IMG_BY_ID, VAL_GT_BY_ID = build_validation_index(VALIDATION_DIR)
val_images = [load_image_rgb(VAL_IMG_BY_ID[i]) for i in val_ids]
val_gts = [load_masks(VAL_GT_BY_ID[i]) for i in val_ids]

eqdiam, areas = [], []
for i, g in zip(val_ids, val_gts):
    rps = regionprops(g)
    d = [r.equivalent_diameter for r in rps]
    a = [r.area for r in rps]
    eqdiam.extend(d)
    areas.extend(a)
    print(f"{i:<8} objects = {len(rps):>3} | mean eq. diameter = {np.mean(d):7.2f} px | min area = {np.min(a):6.0f} px²")
eqdiam, areas = np.array(eqdiam), np.array(areas)

DIAMETER = float(eqdiam.mean())
MIN_SIZE = int(np.floor(areas.min()))

print(f"\nObjects: {len(areas)} in {len(val_ids)} images")
print(f"DIAMETER = {DIAMETER:.2f} px (mean equivalent diameter)")
print(f"MIN_SIZE = {MIN_SIZE} px² (smallest annotated area)")

save_json(dict(diameter=DIAMETER, min_size=MIN_SIZE, n_objects=len(areas), validation_images=val_ids,
               eqdiam_px=dict(mean=eqdiam.mean(), median=np.median(eqdiam), min=eqdiam.min(), max=eqdiam.max()),
               area_px=dict(mean=areas.mean(), median=np.median(areas), min=areas.min())),
          RESULTS_DIR / "phase3_validation_morphology.json")

## 6. Phase 2 model registry

In [ ]:
def find_model(condition, fold):
    matches = sorted((MODELS_DIR / condition).glob(f"{condition}*fold_{fold}_*.pth"))
    if not matches:
        raise FileNotFoundError(f"No model for {condition} fold {fold} in {MODELS_DIR / condition}")
    return matches[0]


registry = []
for condition in CONDITIONS:
    with open(PHASE2_DIR / condition / "results_final.json") as f:
        for rec in json.load(f):
            rec["_condition"] = condition
            rec["_model_path"] = find_model(condition, rec["fold"])
            registry.append(rec)

for r in registry:
    print(f"{r['_condition']:20s} fold {r['fold']} | test = {r['test_image_names']} | {r['_model_path'].name}")

## 7. Validation grid search (`flow_threshold` × `cellprob_threshold`)

In [ ]:
def load_model(model_path):
    return models.CellposeModel(gpu=True, pretrained_model=str(model_path))


def eval_model(model, images, ft, cpt):
    out = model.eval(images, diameter=DIAMETER, flow_threshold=ft, cellprob_threshold=cpt,
                     min_size=MIN_SIZE, channel_axis=CHANNEL_AXIS, normalize=True)
    masks = out[0]
    if not isinstance(masks, list):
        masks = [masks]
    return [np.asarray(m).astype(np.int32) for m in masks]


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def pick_best(combos):
    return max(combos, key=lambda c: (c["aggregate"]["AP@0.5"]["AP"],
                                      c["aggregate"]["AP@0.75"]["AP"],
                                      c["aggregate"]["AP@0.5"]["F1"]))


grid = [(ft, cpt) for ft in FLOW_THRESHOLDS for cpt in CELLPROB_THRESHOLDS]
validation_report = []
for r in tqdm(registry, desc="models"):
    t0 = time.time()
    model = load_model(r["_model_path"])
    combos, per_by_combo = [], {}
    for ft, cpt in tqdm(grid, desc=f"{r['_condition']} fold {r['fold']}", leave=False):
        agg, per = compute_metrics(val_gts, eval_model(model, val_images, ft, cpt))
        combos.append(dict(flow_threshold=ft, cellprob_threshold=cpt, aggregate=agg))
        per_by_combo[(ft, cpt)] = per

    best = pick_best(combos)
    validation_report.append(dict(
        condition=r["_condition"], fold=r["fold"], experiment=r["experiment"],
        use_augmentation=r["use_augmentation"], model_name=r["model_name"],
        model_file=r["_model_path"].name,
        fixed_inference=dict(diameter=DIAMETER, min_size=MIN_SIZE, channel_axis=CHANNEL_AXIS),
        grid=dict(flow_thresholds=FLOW_THRESHOLDS, cellprob_thresholds=CELLPROB_THRESHOLDS),
        validation_image_names=val_ids,
        best_flow_threshold=best["flow_threshold"],
        best_cellprob_threshold=best["cellprob_threshold"],
        best_validation_metrics=best["aggregate"],
        best_validation_per_image=per_by_combo[(best["flow_threshold"], best["cellprob_threshold"])],
        all_combos=combos,
        gridsearch_time_seconds=time.time() - t0))
    print(f"{r['_condition']} fold {r['fold']}: flow = {best['flow_threshold']}, "
          f"cellprob = {best['cellprob_threshold']}, validation AP@50 = {best['aggregate']['AP@0.5']['AP']:.3f}")
    del model
    free_gpu()

save_json(validation_report, RESULTS_DIR / "phase3_validation_gridsearch.json")

In [ ]:
grid_df = pd.DataFrame([
    dict(condition=e["condition"], fold=e["fold"], flow=c["flow_threshold"], cellprob=c["cellprob_threshold"],
         AP50=c["aggregate"]["AP@0.5"]["AP"], AP75=c["aggregate"]["AP@0.75"]["AP"],
         AP90=c["aggregate"]["AP@0.9"]["AP"], F1=c["aggregate"]["AP@0.5"]["F1"])
    for e in validation_report for c in e["all_combos"]
])
grid_df.to_csv(RESULTS_DIR / "phase3_validation_grid.csv", index=False)

best_df = pd.DataFrame([dict(condition=e["condition"], fold=e["fold"], flow=e["best_flow_threshold"],
                             cellprob=e["best_cellprob_threshold"],
                             val_AP50=e["best_validation_metrics"]["AP@0.5"]["AP"],
                             val_AP75=e["best_validation_metrics"]["AP@0.75"]["AP"],
                             val_AP90=e["best_validation_metrics"]["AP@0.9"]["AP"])
                        for e in validation_report])
print(best_df.to_string(index=False))
print()
print(best_df.groupby("condition")[["val_AP50", "val_AP75", "val_AP90"]].agg(["mean", "std"]).round(3))

## 8. Test-fold re-evaluation with the selected parameters

In [ ]:
best_params = {(e["condition"], e["fold"]): (e["best_flow_threshold"], e["best_cellprob_threshold"])
               for e in validation_report}
val_ap50 = {(e["condition"], e["fold"]): e["best_validation_metrics"]["AP@0.5"]["AP"]
            for e in validation_report}

test_report = []
for r in tqdm(registry, desc="models"):
    key = (r["_condition"], r["fold"])
    ft, cpt = best_params[key]

    imgs, gts, used = [], [], []
    for n in [str(n) for n in r["test_image_names"]]:
        ip, gp = resolve_cv_pair(n, CV_DIRS)
        if ip is None or gp is None:
            print(f"WARNING: missing files for image {n}")
            continue
        imgs.append(load_image_rgb(ip))
        gts.append(load_masks(gp))
        used.append(n)

    t0 = time.time()
    model = load_model(r["_model_path"])
    t_load = time.time() - t0
    t0 = time.time()
    preds = eval_model(model, imgs, ft, cpt)
    t_eval = time.time() - t0
    agg, per = compute_metrics(gts, preds)

    test_report.append(dict(
        fold=r["fold"], experiment=r["experiment"], condition=r["_condition"],
        model_name=r["model_name"], model_file=r["_model_path"].name,
        use_augmentation=r["use_augmentation"],
        hyperparameters=dict(**TRAIN_HP, augmentation_multiplier=r["hyperparameters"].get("augmentation_multiplier", 0)),
        inference_parameters=dict(diameter=DIAMETER, min_size=MIN_SIZE, flow_threshold=ft,
                                  cellprob_threshold=cpt, channel_axis=CHANNEL_AXIS),
        selected_on_validation=dict(flow_threshold=ft, cellprob_threshold=cpt, validation_AP50=val_ap50[key]),
        n_train=r["n_train"], n_test=len(used), test_image_names=used,
        model_load_time_seconds=t_load, evaluation_time_seconds=t_eval,
        aggregate_metrics=agg, per_image_metrics=per))
    print(f"{r['_condition']} fold {r['fold']}: test AP@50 = {agg['AP@0.5']['AP']:.3f}, F1 = {agg['AP@0.5']['F1']:.3f}")
    del model
    free_gpu()

save_json(test_report, RESULTS_DIR / "phase3_test_evaluation.json")

## 9. Comparison of conditions after inference optimization

In [ ]:
def fold_values(cond, thr="AP@0.5", field="AP"):
    return np.array([e["aggregate_metrics"][thr][field] for e in test_report if e["condition"] == cond])


summary_rows = []
for cond in CONDITIONS:
    for thr in IOU_THRESHOLDS:
        for field in ("AP", "Precision", "Recall", "F1"):
            v = fold_values(cond, f"AP@{thr}", field)
            summary_rows.append(dict(condition=cond, metric=f"AP@{thr}", field=field, mean=v.mean(),
                                     std=v.std(ddof=1), se=v.std(ddof=1) / np.sqrt(len(v)), n_folds=len(v)))
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(RESULTS_DIR / "phase3_augmentation_summary.csv", index=False)
print(summary_df.round(3).to_string(index=False))

In [ ]:
a = fold_values("A_no_augmentation")
b = fold_values("B_with_augmentation")
better = a if a.mean() >= b.mean() else b
se_best = better.std(ddof=1) / np.sqrt(len(better))

print(f"AP@50 without augmentation = {a.mean():.3f} ± {a.std(ddof=1):.3f}")
print(f"AP@50 with augmentation    = {b.mean():.3f} ± {b.std(ddof=1):.3f}")
print(f"One-SE threshold (best condition) = {better.mean() - se_best:.3f}")
if abs(a.mean() - b.mean()) <= se_best:
    print("Difference within one SE: statistically equivalent.")
else:
    print(f"Difference exceeds one SE in favour of {'augmentation' if b.mean() > a.mean() else 'no augmentation'}.")

## 10. Change relative to Phase 2 (default inference)

In [ ]:
delta_rows = []
for e in test_report:
    ph2 = next(r for r in registry if r["_condition"] == e["condition"] and r["fold"] == e["fold"])
    for thr in IOU_THRESHOLDS:
        key = f"AP@{thr}"
        before = ph2["aggregate_metrics"][key]["AP"]
        after = e["aggregate_metrics"][key]["AP"]
        delta_rows.append(dict(condition=e["condition"], fold=e["fold"], metric=key,
                               phase2_default=before, phase3_optimized=after, delta=after - before))
delta_df = pd.DataFrame(delta_rows)
delta_df.to_csv(RESULTS_DIR / "phase3_vs_phase2_delta.csv", index=False)
print(delta_df.pivot_table(index=["condition", "fold"], columns="metric", values="delta").round(3).to_string())
print()
print(delta_df.groupby(["condition", "metric"])["delta"].mean().round(3).to_string())